# 12. Pipeline Architecture & Performance Scaling (5+ Years Interview Guide)
Exhaustive revision guide to method chaining with .pipe(), chunked processing on raw_transactions.csv, and PyArrow backend scaling.

### Key 5-Year Interview Concepts Covered:
- **Method Chaining Architecture**: Dedicated cell for `.pipe(custom_function)`.
- **Chunked File Iteration**: Dedicated cell for `pd.read_csv(filepath, chunksize=10000)`.

This interactive revision guide uses `data/raw_transactions.csv` for all real-world code examples.

In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### Method Chaining with `.pipe()`
**Explanation**: Constructs a clean transformation pipeline on raw transactions.

**Syntax**: `df.pipe(clean_dates).pipe(filter_fraud).pipe(calculate_metrics)`

In [2]:
def filter_valid_transactions(data):
    return data.dropna(subset=['transaction_amount'])

def add_fraud_flag_str(data):
    return data.assign(fraud_label=np.where(data['is_fraud'] == 1, 'FRAUD', 'LEGIT'))

clean_pipeline = df.pipe(filter_valid_transactions).pipe(add_fraud_flag_str)
print('Pipeline Processed Transactions Head:\n', clean_pipeline[['transaction_id', 'transaction_amount', 'fraud_label']].head(3))

Pipeline Processed Transactions Head:
   transaction_id  transaction_amount fraud_label
0       TX110686             1216.33       LEGIT
1       TX107170              324.99       LEGIT
2       TX108328              136.66       LEGIT


### Chunked Processing with `pd.read_csv(chunksize=...)`
**Explanation**: Streams `data/raw_transactions.csv` in 2,500-row chunks to calculate global metrics without memory spikes.

**Syntax**: `for chunk in pd.read_csv('data/raw_transactions.csv', chunksize=2500): ...`

In [3]:
total_spend = 0.0
total_tx_count = 0
for chunk in pd.read_csv(csv_path, chunksize=2500):
    total_spend += chunk['transaction_amount'].sum()
    total_tx_count += len(chunk)
print(f'Processed {total_tx_count} transactions across chunks. Total Revenue: ${total_spend:,.2f}')

Processed 15000 transactions across chunks. Total Revenue: $14,349,538.14


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: PyArrow Engine Zero-Copy Performance
**Explanation**: Ingest `raw_transactions.csv` using the PyArrow engine (`dtype_backend='pyarrow'`) for multi-threaded parsing.

**Syntax**: `pd.read_csv('data/raw_transactions.csv', engine='pyarrow', dtype_backend='pyarrow')`

In [4]:
t0 = time.perf_counter()
df_arrow = pd.read_csv(csv_path, engine='pyarrow', dtype_backend='pyarrow')
t_arrow = time.perf_counter() - t0
print(f'PyArrow Ingestion Time: {t_arrow*1000:.2f} ms ({df_arrow.shape[0]} rows)')

PyArrow Ingestion Time: 6.94 ms (15000 rows)
